<a href="https://colab.research.google.com/github/fotomain/py_labs_ai_crash_course/blob/main/CAPSTONE_V1_OR_TOOLS1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install "pandas==2.2.2" "numpy<2.1,>=1.22" "protobuf>=5.26.1,<6.0.0dev" "ortools==9.12.4544"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.9/60.9 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 41.6 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.5.1
    Uninstalling numpy-2.5.1:
      Successfully uninstalled numpy-2.5.1
  Attempting uninstall: pandas
    Found existing installation: pandas 3.0.3
    Uninstalling pandas-3.0.3:
      Successfully uninstalled pandas-3.0.3


In [3]:
# Install dependencies if needed
!pip install prophet gdown pandas numpy

import pandas as pd
import numpy as np
from prophet import Prophet
import gdown
import os
import warnings
warnings.filterwarnings('ignore')

# ------------------------------------------------------------
# 0. Download data
# ------------------------------------------------------------
def download_data_from_drive():
    file_id = '1lWe3IB8L_3TpDHO1MRKDqaH1Tk20pzDk'
    url = f'https://drive.google.com/uc?id={file_id}'
    output = 'sales_data.csv'
    if not os.path.exists(output):
        print('[0] Downloading sales_data.csv...')
        gdown.download(url, output, quiet=False)
    else:
        print('[0] File already exists.')
    return output

# ------------------------------------------------------------
# 1. Load data
# ------------------------------------------------------------
def load_data(filepath):
    data = pd.read_csv(filepath)
    data['Month'] = pd.to_datetime(data['Month'])
    data.set_index('Month', inplace=True)
    return data

# ------------------------------------------------------------
# 2. Forecast with Prophet
# ------------------------------------------------------------
def forecast_sales(data, periods=24):
    df = data.rename(columns={'#Passengers': 'y'})
    df['ds'] = df.index
    df = df[['ds', 'y']]
    model = Prophet(yearly_seasonality=True, weekly_seasonality=False)
    model.fit(df)
    future = model.make_future_dataframe(periods=periods, freq='MS')
    forecast = model.predict(future)
    forecast_1961 = forecast[forecast['ds'].dt.year == 1961]
    predicted = pd.DataFrame({
        'Month': forecast_1961['ds'].dt.strftime('%Y-%m'),
        'Predicted_Passengers': np.round(forecast_1961['yhat']).astype(int)
    })
    predicted['Month'] = pd.to_datetime(predicted['Month'])
    predicted.set_index('Month', inplace=True)
    return predicted, model

# ------------------------------------------------------------
# 3. Generate weekly orders (heuristic)
# ------------------------------------------------------------
def generate_orders_heuristic(predicted_plan, max_order=100, safety_stock_factor=0.2):
    """
    Convert monthly demand to weekly orders.
    Each order ≤ max_order.
    Returns DataFrame with columns:
        week, month, order_qty, demand, inventory_after
    """
    orders = []
    stock = 0.0
    week_counter = 0

    for month, row in predicted_plan.iterrows():
        monthly_demand = row['Predicted_Passengers']
        weekly_demand = monthly_demand / 4.0
        safety_stock = weekly_demand * safety_stock_factor

        for week_in_month in range(1, 5):
            week_counter += 1

            needed = max(0.0, weekly_demand + safety_stock - stock)
            if needed > 0:
                order_qty = min(max_order, needed)
                stock += order_qty
            else:
                order_qty = 0

            stock -= weekly_demand
            stock = max(0.0, stock)

            orders.append({
                'week': week_counter,
                'month': month.strftime('%Y-%m'),
                'order_qty': int(order_qty),
                'demand': round(weekly_demand, 1),
                'inventory_after': round(stock, 1)
            })

    return pd.DataFrame(orders)

# ------------------------------------------------------------
# 4. Save outputs
# ------------------------------------------------------------
def save_outputs(predicted_plan, order_df):
    predicted_plan.to_csv('predicted_plan.csv')
    print('✓ Saved predicted_plan.csv')

    order_df['month_num'] = (order_df['week'] - 1) // 4 + 1
    for m in range(1, 13):
        month_data = order_df[order_df['month_num'] == m]
        month_data[['week', 'order_qty', 'demand', 'inventory_after']].to_csv(
            f'order_month_{m:02d}.csv', index=False
        )
        print(f'✓ Saved order_month_{m:02d}.csv')

# ------------------------------------------------------------
# 5. Main
# ------------------------------------------------------------
def main():
    print('=' * 70)
    print(' SALES FORECASTING + WEEKLY ORDER PLANNING')
    print(' (Heuristic – max 100 units per order)')
    print('=' * 70)

    csv_file = download_data_from_drive()
    print('\n[1] Loading sales data...')
    data = load_data(csv_file)
    print(f'    Loaded {len(data)} monthly records.')

    print('\n[2] Forecasting sales for 1961...')
    predicted_plan, _ = forecast_sales(data)
    print(f'    Forecast generated for {len(predicted_plan)} months.')

    print('\n[3] Generating weekly orders...')
    order_df = generate_orders_heuristic(predicted_plan)
    print(f'    Generated {len(order_df)} weekly orders.')

    print('\n[4] Saving output files...')
    save_outputs(predicted_plan, order_df)

    total_orders = order_df['order_qty'].gt(0).sum()
    total_units = int(order_df['order_qty'].sum())
    avg_order = order_df['order_qty'].mean()

    print('\n' + '=' * 70)
    print(' SUMMARY STATISTICS')
    print('=' * 70)
    print(f'Total Annual Demand            : {int(predicted_plan["Predicted_Passengers"].sum()):,} units')
    print(f'Total Orders Placed            : {total_orders}')
    print(f'Total Units Ordered            : {total_units:,} units')
    print(f'Average Order Quantity         : {avg_order:.1f} units')
    print(f'Maximum Order Quantity         : {int(order_df["order_qty"].max())} units')

    print('\nPeak Demand Months (Top 3):')
    peak = predicted_plan.nlargest(3, 'Predicted_Passengers')
    for month, row in peak.iterrows():
        print(f'    {month.strftime("%B %Y")}: {int(row["Predicted_Passengers"])} passengers')

    print('\n' + '=' * 70)
    print('✅ Process completed successfully!')
    print('=' * 70)

if __name__ == '__main__':
    main()

INFO:prophet:Disabling daily seasonality. Run prophet with daily_seasonality=True to override this.


 SALES FORECASTING + WEEKLY ORDER PLANNING
 (Heuristic – max 100 units per order)
[0] File already exists.

[1] Loading sales data...
    Loaded 144 monthly records.

[2] Forecasting sales for 1961...
    Forecast generated for 12 months.

[3] Generating weekly orders...
    Generated 48 weekly orders.

[4] Saving output files...
✓ Saved predicted_plan.csv
✓ Saved order_month_01.csv
✓ Saved order_month_02.csv
✓ Saved order_month_03.csv
✓ Saved order_month_04.csv
✓ Saved order_month_05.csv
✓ Saved order_month_06.csv
✓ Saved order_month_07.csv
✓ Saved order_month_08.csv
✓ Saved order_month_09.csv
✓ Saved order_month_10.csv
✓ Saved order_month_11.csv
✓ Saved order_month_12.csv

 SUMMARY STATISTICS
Total Annual Demand            : 6,074 units
Total Orders Placed            : 48
Total Units Ordered            : 4,800 units
Average Order Quantity         : 100.0 units
Maximum Order Quantity         : 100 units

Peak Demand Months (Top 3):
    August 1961: 578 passengers
    July 1961: 577 pa